### Gym Gridworld - RL

In [2]:
import time
import gymnasium
from gymnasium.wrappers import FlattenObservation, HumanRendering, RecordEpisodeStatistics, RecordVideo
import numpy as np

from src.environments.Gridworld import GridWorldEnv, DiscreteGridWorldWrapper, NormalizedCoordWrapper, PatienceWrapper, Cells

from src.agents.QLambdaAgent import QLambdaAgent
from src.agents.DQNAgent import DQNAgent
from src.agents.BasicDQNAgent import BasicDQNAgent

from src.solvers.GridWorldSolver import GridWorldSolver
from src.solvers.BasicGridWorldSolver import BasicGridWorldSolver

from src.utility.ReplayBuffers import DequeReplayBuffer, PrioritizedReplayBuffer
from src.utility.SessionRunner import play_game

### Grids

In [3]:
medium_grid = np.array((
    [0,0,0,0,0,2],
    [0,3,3,3,0,0],
    [0,0,0,0,0,0],
    [0,2,3,0,3,0],
    [0,0,3,1,3,0],
    [0,0,0,0,0,0],
))

huge_grid = np.array((
    [0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0,0,0,0,0,0,0,3,0,0,0,0],
    [0,0,0,0,0,0,3,0,3,3,3,3,3,0,0,0,0,3,0,0,0,0,0,0,0,3,0,0,0,3,0,0],
    [0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,3,0,3,0,3,0,0],
    [0,0,0,0,0,0,3,0,0,0,0,0,3,0,0,3,3,3,0,0,0,0,0,0,0,3,3,3,3,3,0,0],
    [0,0,0,0,0,0,3,3,3,3,3,0,3,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,3,0,0],
    [0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,3,3,3,3,3,0,0,0,0,0,0,0,3,0,0],
    [0,0,0,0,0,0,0,3,3,3,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0],
    [0,0,0,0,0,0,0,3,0,3,0,0,3,0,3,3,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0],
    [0,0,0,0,0,0,0,3,0,3,3,3,3,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0],
    [0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0],
    [0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,0,0,3,3,3,3,3,0,0,0,0,0,0,3,0,0],
    [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,3,3,3,3,3,0,0,0,3,3,3,3,0,0],
    [3,3,3,3,0,3,3,3,3,3,3,3,3,3,0,3,0,0,3,3,3,3,3,0,0,0,3,0,0,0,0,0],
    [3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,2,2,3,3,3,3,3,0,0,0,3,0,0,0,0,0],
    [3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,3,3,3,3,3,0,0,0,3,0,0,1,0,0],
    [3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,3,3,3,3,3,0,0,0,3,0,0,0,0,0],
    [3,3,0,3,3,0,3,3,3,0,3,3,3,3,3,3,0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,0],
    [0,0,0,0,0,0,0,3,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
    [0,0,2,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0]
))

### Learning how to set up the environment

Cound create environment directly like this:  
```py
env = GridWorldEnv()
``` 
However, the environment is registered in gymnasium in the __init__.py files in *environments*, so it can also be created via *gymnasium.make*

In [ ]:
# Custom reward structure
huge_world_reward: dict = {
    Cells.TILE.value: -0.005, # step on regular cell
    Cells.TARGET.value: 1000.0, # good ending
    Cells.PITFALL.value: -100.0, # bad ending
    Cells.WALL.value: -2.0 # hit obstacle
} # Adjusting the rewards can be considered hyperparameter tuning - we can help the mode converge faster by representing the reward structure better

# Environment
env = gymnasium.make(id="gymnasium_env/GridWorld-v0", grid=huge_grid, reward=huge_world_reward, render_mode=None)
# or env = GridWorldEnv(grid=huge_grid, reward=huge_world_reward, render_mode=None)

In [3]:
env.observation_space

Dict('agent': Box(0, [19 32], (2,), int64))

In [5]:
obs, info = env.reset()
print("Observation: ", obs)
print("Info: ", info)

Observation:  {'agent': array([11,  4])}
Info:  {'distance': np.float64(28.0)}


In [7]:
# Wrappers can be used on the env for various purposes, like flattening observations into a single array
wrapped_env = FlattenObservation(env)
wrapped_env.observation_space

Box(0, [19 32], (2,), int64)

This is particularly useful when working with algorithms that expect specific input formats (like neural networks that need 1D arrays instead of dictionaries).

In [ ]:
obs, info = wrapped_env.reset()
print("Observation: ", obs)

Observation:  [ 4 15]


: 

### Training Agent - Q Lambda

In [ ]:
# Custom reward structure
huge_world_reward: dict = {
    Cells.TILE.value: -0.5, # step on regular cell
    Cells.TARGET.value: 1000.0, # good ending
    Cells.PITFALL.value: -100.0, # bad ending
    Cells.WALL.value: -2.0 # hit obstacle
} # Adjusting the rewards can be considered hyperparameter tuning - we can help the mode converge faster by representing the reward structure better

In [4]:
env = GridWorldEnv(grid=huge_grid, reward=huge_world_reward, wind_p=0.2, render_mode="rgb_array")
# Create a wrapper that maps gridworld position to state integer
wrapped_env = DiscreteGridWorldWrapper(env)

In [8]:
q_lambda_learner_kwargs = {
    "gamma": 0.95, # discount factor for Q' when computing delta ( delta = r + gamma * Q' - Q(s, a) )
    "alpha": 0.01, # 'learning rate' - affects step size in direction of delta when updating Q
    "epsilon": 1.0, # exploration threshold
    "xi": 0.99, # exploration threshold decay
    "lam": 0.6 # λ parameter ; Reflects how far back credit assignment goes
}

In [9]:
solver = GridWorldSolver(wrapped_env, QLambdaAgent, q_lambda_learner_kwargs, verbose=True)

In [ ]:
solver.train(n_epochs=5000)

Epoch: 100%|██████████| 5000/5000 [01:06<00:00, 75.07it/s] 

Total reward = 2657499.0


In [12]:
# use a wrapper to create a human-renderable env from the original
human_env = HumanRendering(wrapped_env)

In [13]:
play_game(solver.agent, human_env, num_episodes=1)

--- Episode 1 Starting ---
Episode Finished. Num steps: 28; Total reward: 983.50


### Training Agent - Basic DQN

In [ ]:
(grid, reward) = (
    # --- Grid---
    huge_grid,
    # --- Reward ---
    {
        Cells.TILE.value: -0.05, # step on regular cell
        Cells.TARGET.value: 200.0, # good ending
        Cells.PITFALL.value: -50.0, # bad ending
        Cells.WALL.value: -1.0 # hit obstacle
    }
)

# --- Env ---
env = GridWorldEnv(grid=grid, reward=reward, wind_p=0.0, step_limit=500, render_mode="rgb_array")
wrapped_env = NormalizedCoordWrapper(env)
record_video_every_n = 100
wrapped_env = RecordVideo(
    wrapped_env,
    video_folder=f"../runtime/videos/trial_{time.strftime('%X_%x').replace(':', '').replace('/', '')}",
    episode_trigger=lambda episode_id: episode_id % record_video_every_n == 0,
    disable_logger=True # Keeps the console clean
)

#  --- Agent ---
replay_buffer = DequeReplayBuffer(capacity=50_000)
basic_dqn_learner_kwargs = {
    "replay_buffer": replay_buffer,
    "gamma": 0.95, # discount factor for Q' when computing delta ( delta = r + gamma * Q' - Q(s, a) )
    "alpha": 5e-3,
    "epsilon": 1.0, # exploration threshold
    "xi": 0.995, # exploration threshold decay
    # QNetwork
    "hidden_dim": 128,  
    "batch_size": 64,
    "target_update_freq": 200,
    "train_every_n": 4,
}

# --- Solver ---
basic_dqn_solver = BasicGridWorldSolver(wrapped_env, BasicDQNAgent, basic_dqn_learner_kwargs, verbose=True)

# --- Train ---
basic_dqn_solver.train(n_epochs=2000)

Epoch:  41%|████▏     | 827/2000 [03:37<05:08,  3.80it/s]


KeyboardInterrupt: 

In [6]:
human_env = HumanRendering(wrapped_env)

In [ ]:
play_game(basic_dqn_solver.agent, human_env)

--- Episode 1 Starting ---
Episode Finished. Num steps: 51; Total reward: -51.00


: 

### Training Agent - DQN

In [4]:
(grid, reward) = (
    # --- Grid---
    huge_grid,
    # --- Reward ---
    {
        Cells.TILE.value: -0.05, # step on regular cell
        Cells.TARGET.value: 200.0, # good ending
        Cells.PITFALL.value: -50.0, # bad ending
        Cells.WALL.value: -1.0 # hit obstacle
    }
)

# --- Env ---
env = GridWorldEnv(grid=grid, reward=reward, wind_p=0.0, step_limit=500, render_mode="rgb_array")
wrapped_env = PatienceWrapper(env)
wrapped_env = NormalizedCoordWrapper(wrapped_env)
record_video_every_n = 100
wrapped_env = RecordVideo(
    wrapped_env,
    video_folder=f"../runtime/videos/trial_{time.strftime('%X_%x').replace(':', '').replace('/', '')}",
    episode_trigger=lambda episode_id: episode_id % record_video_every_n == 0,
    disable_logger=True # Keeps the console clean
)

#  --- Agent ---
dqn_learner_kwargs = {
    "gamma": 0.95, # discount factor for Q' when computing delta ( delta = r + gamma * Q' - Q(s, a) )
    "alpha": 5e-4,
    "epsilon": 1.0, # exploration threshold
    "xi": 0.995, # exploration threshold decay
    "warmup_episodes": 50, # warmup counter - number of episodes before xi gets enabled to start decaying epsilon (exporation threshold)
    "r_scaling": 1.0 / env.get_max_abs_reward(),
    # QNetwork
    "hidden_dim": 128,  
    "batch_size": 64,
    "target_update_freq": 200,
    "train_every_n": 4,
}

# --- Solver ---
dqn_solver = GridWorldSolver(wrapped_env, DQNAgent, dqn_learner_kwargs, verbose=True)

# --- Train ---
dqn_solver.train(n_epochs=2000)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: craiucalin (craiucalin-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch: 100%|██████████| 2000/2000 [00:31<00:00, 63.11it/s]


epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇████
train/epsilon,█▆▆▅▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/grad_norm_clipped,▃▅▄▃▂▄▅▁▃▂▄▅▂█▄▁▂▂▂▂▃▃▂▁▂▃▂▂▂▂▄▃▄▂▄▂▃▂▄▂
train/grad_norm_unclipped,▃▄▁▂▄▂▂▂▃▁▇▄▄▃▃▄█▅▄▂▂▁▄▄▂▂▄▃▅▃▃▂▂▄▃▂▂▅▃▁
train/loss,▄▄▅▂▄▄▅▂▃█▂▂▂▄▂▂▁▁▂▂▂▁▁▁▁▁▁▁▁▁▁▂▂▁▃▂▁▁▁▁
train/max_q,▁▂▃▃▃▄▅▅▅▆▆▆▆▅▇▇▇▇▇▇▇▇▇▇▆▇████▇▇▇▇▇▆▇█▇▇
train/mean_q,▁▃▄▄▅▅▆█▆▆▅▇▆▇▆▇▆█▆▇█▆▆▇▆▆▆█▆▆▆██▅▇▆▄█▇▅
train/param_norm,▁▂▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇█████
train/q_std,▁▂▂▃▂▁▅▃▄▄▅▅▆▆▇▄▅▅▅▄▅█▇▅▇▇▆▆▇▅▆█▇▆▄▇▄▇▄▅
train/td_error,█▃▄▄▅▃▄▅▃▅▃▃▃▂▃▃▂▂▂▁▂▂▂▃▃▂▅▃▂▃▂▂▁▁▂▃▂▂▂▄
epoch,1999


In [4]:
human_env = HumanRendering(wrapped_env)

In [ ]:
play_game(dqn_solver.agent, human_env, num_episodes=1)

--- Episode 1 Starting ---
Episode Finished. Num steps: 6; Total reward: -5.05


: 